In [1]:
# check installed version
import pycaret
pycaret.__version__

'3.0.0'

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import PredefinedSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

from pycaret.classification import ClassificationExperiment
from datetime import datetime
import pytz

1.data

In [ ]:
import pandas as pd
from pycaret.classification import setup, compare_models, pull
from pycaret.classification import *

gene_expression_df=pd.read_csv('/home/chenliqun/Tcga/input/newtop50_matrix_mincancer1.txt',sep='\t',index_col=0)
X=gene_expression_df.transpose()  # shape=(2959,219) 144
print("X shape:",X.shape)

labels_df=pd.read_csv('/home/chenliqun/Tcga/input/Final_TCGA_sourcetumornames_selected7tumors_TcgaTargetGTEX_phenotype_filtered_samples.txt',sep='\t')
labels_df.set_index('sample',inplace=True)
y=labels_df['sourced_tumor_name']

# Ensure y follows the same index order as X
y=y.reindex(X.index)

# Check if all samples have labels
missing_labels=y.isnull().sum()
if missing_labels>0:
    print(f"Warning: {missing_labels} samples have no corresponding labels in the label file")
    X=X[y.notnull()]
    y=y.dropna()

data=X.copy()
data['target']=y.values  # Add target column

print(f"Merged data shape: {data.shape}")
print(f"Number of features: {data.shape[1]-1}")
print(f"Number of target classes: {data['target'].nunique()}")
print("Class distribution:")
print(data['target'].value_counts())

# Now 'data' is in the format required by PyCaret

X形状: (2959, 144)
合并后的数据形状: (2959, 145)
特征数量: 144
目标类别数量: 7
类别分布:
target
ER positive breast cancer           1092
Clear cell renal cell carcinoma      530
Bladder urothelial carcinoma         407
Colorectal cancer                    380
Cervical squamous cell carcinoma     304
Endometrial carcinoma                180
Chromophobe renal cell carcinoma      66
Name: count, dtype: int64


In [31]:
SESSION_ID = 2026

In [ ]:
exp = setup(
    data=data,
    target='target',
    train_size=0.8,  
    session_id=SESSION_ID,   
    fold=5,         
    data_split_stratify=True,
    verbose=True,
    normalize=True,  
)

setup_info = pull()
print("\nExperiment setup completed!")
print(f"Dataset samples: {len(exp.dataset)}")
print(f"Training samples: {len(exp.train)}")
print(f"Test samples: {len(exp.test)}")

,Description,Value
0,Session id,2026
1,Target,target
2,Target type,Multiclass
3,Target mapping,"Bladder urothelial carcinoma: 0, Cervical squamous cell carcinoma: 1, Chromophobe renal cell carcinoma: 2, Clear cell renal cell carcinoma: 3, Colorectal cancer: 4, ER positive breast cancer: 5, Endometrial carcinoma: 6"
4,Original data shape,"(2959, 145)"
5,Transformed data shape,"(2959, 145)"
6,Transformed train set shape,"(2367, 145)"
7,Transformed test set shape,"(592, 145)"
8,Numeric features,144
9,Preprocess,True



实验设置完成！
训练集样本数: 2959
训练集样本数: 2367
测试集样本数: 592


In [ ]:
train_data = exp.train
test_data = exp.test
print(f"\nTraining set: {train_data.shape}")
print(f"Test set: {test_data.shape}")
train_data.to_csv('mincancer1_train_data.csv')
test_data.to_csv('mincancer1_test_data.csv')
print("Training and test sets saved to mincancer1_train_data.csv and mincancer1_test_data.csv")


训练集: (2367, 145)
测试集: (592, 145)
训练集和测试集已保存到 train_data.csv 和 test_data.csv


In [ ]:
print(f"now session_id: {exp.seed}")

当前session_id: 2026


In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score
from pycaret.classification import *
add_metric(
    id="bal_acc",
    name="bal_acc",                      
    score_func=balanced_accuracy_score,
    greater_is_better=True
)
add_metric(
    id="f1_macro",
    name="f1_macro",                     
    score_func=lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro"),
    greater_is_better=True
)

top16_model = compare_models(
    n_select=16,
    sort="Accuracy",
    fold=5,
    verbose=True
)

cv_results = pull()
print(cv_results)

cv_results.to_csv("mincancer1_cv_results.csv", index=False, encoding="utf-8-sig")

print("\n best:")
print(top16_model)
print("\n Saved：mincancer1_cv_results.csv")


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,bal_acc,f1_macro,TT (Sec)
ridge,Ridge Classifier,0.9759,0.0000,0.9759,0.9770,0.9760,0.9692,0.9693,0.9642,0.0000,0.4160
lda,Linear Discriminant Analysis,0.9742,0.0000,0.9742,0.9761,0.9746,0.9671,0.9672,0.9646,0.0000,0.4280
catboost,CatBoost Classifier,0.9717,0.9988,0.9717,0.9727,0.9717,0.9638,0.9639,0.9585,0.0000,18.1920
lr,Logistic Regression,0.9692,0.0000,0.9692,0.9699,0.9691,0.9606,0.9607,0.9485,0.0000,1.7660
et,Extra Trees Classifier,0.9675,0.9982,0.9675,0.9681,0.9673,0.9584,0.9585,0.9474,0.0000,0.4500
xgboost,Extreme Gradient Boosting,0.9654,0.9986,0.9654,0.9658,0.9651,0.9556,0.9557,0.9408,0.0000,0.7980
svm,SVM - Linear Kernel,0.9649,0.0000,0.9649,0.9658,0.9646,0.9551,0.9553,0.9379,0.0000,0.4160
lightgbm,Light Gradient Boosting Machine,0.9645,0.9984,0.9645,0.9652,0.9645,0.9546,0.9547,0.9464,0.0000,915.3120
knn,K Neighbors Classifier,0.9624,0.9915,0.9624,0.9639,0.9625,0.9519,0.9520,0.9472,0.0000,0.4300
rf,Random Forest Classifier,0.9624,0.9981,0.9624,0.9633,0.9622,0.9519,0.9520,0.9422,0.0000,0.5160


Processing:   0%|          | 0/84 [00:00<?, ?it/s]

                                    Model  Accuracy     AUC  Recall   Prec.  \
ridge                    Ridge Classifier    0.9759  0.0000  0.9759  0.9770   
lda          Linear Discriminant Analysis    0.9742  0.0000  0.9742  0.9761   
catboost              CatBoost Classifier    0.9717  0.9988  0.9717  0.9727   
lr                    Logistic Regression    0.9692  0.0000  0.9692  0.9699   
et                 Extra Trees Classifier    0.9675  0.9982  0.9675  0.9681   
xgboost         Extreme Gradient Boosting    0.9654  0.9986  0.9654  0.9658   
svm                   SVM - Linear Kernel    0.9649  0.0000  0.9649  0.9658   
lightgbm  Light Gradient Boosting Machine    0.9645  0.9984  0.9645  0.9652   
knn                K Neighbors Classifier    0.9624  0.9915  0.9624  0.9639   
rf               Random Forest Classifier    0.9624  0.9981  0.9624  0.9633   
gbc          Gradient Boosting Classifier    0.9565  0.0000  0.9565  0.9572   
nb                            Naive Bayes    0.9345 

In [ ]:
from pycaret.classification import *
print("\n" + "="*50)
print("Start saving models...")
print("="*50)

# Save all 16 models
for i, model in enumerate(top16_model, 1):
    model_name = type(model).__name__
    filename = f"model_{i:02d}_{model_name}"
    try:
        save_model(model, filename)
        print(f"Saved model {i:2d}: {model_name} -> {filename}.pkl")
    except Exception as e:
        print(f"Failed to save model {i:2d}: {model_name} - {str(e)}")

print("\n" + "="*50)
print("Model saving completed!")
print("="*50)

# Save the best model (ranked first)
if top16_model:
    best_model = top16_model[0]
    best_model_name = type(best_model).__name__
    try:
        save_model(best_model, "best_model")
        print(f"\nSaved best model: {best_model_name} -> best_model.pkl")
        save_model(best_model, f"best_model_{best_model_name}")
        print(f"Saved best model: {best_model_name} -> best_model_{best_model_name}.pkl")
    except Exception as e:
        print(f"\nFailed to save best model: {str(e)}")

# Save model list information
with open("models_list.txt", "w", encoding="utf-8") as f:
    f.write("Top 16 best models list:\n")
    f.write("=" * 50 + "\n")
    for i, model in enumerate(top16_model, 1):
        model_name = type(model).__name__
        f.write(f"{i:2d}. {model_name}\n")
    f.write("=" * 50 + "\n")

print("\nModel list saved to models_list.txt")

# Create detailed model summary
import pandas as pd
import joblib

models_info = []
for i, model in enumerate(top16_model, 1):
    model_info = {
        "Rank": i,
        "Model Name": type(model).__name__,
        "Model Type": str(type(model)),
        "Feature Count": model.__dict__.get("n_features_in_", "Unknown"),
        "File Path": f"model_{i:02d}_{type(model).__name__}.pkl",
        "Save Status": "Success"
    }
    models_info.append(model_info)

models_df = pd.DataFrame(models_info)
models_df.to_csv("models_summary.csv", index=False, encoding="utf-8-sig")

print("Model details saved to models_summary.csv")

# Create a zip archive containing all models
import zipfile
import os

with zipfile.ZipFile("all_models.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir("."):
        if file.endswith(".pkl"):
            zipf.write(file)
            print(f"Added {file} to archive")

    for file in [
        "models_list.txt",
        "models_summary.csv",
        "cross_validation_results.csv",
        "train_data.csv",
        "test_data.csv"
    ]:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Added {file} to archive")

print("\nAll models packaged into all_models.zip")

# Verify model loading
print("\n" + "="*50)
print("Verifying model loading...")
print("="*50)

if os.path.exists("best_model.pkl"):
    try:
        loaded_best_model = load_model("best_model")
        print("Best model loaded successfully")
        print(f"   Model type: {type(loaded_best_model).__name__}")
        if hasattr(loaded_best_model, "get_params"):
            print("   Parameter example:", str(loaded_best_model.get_params())[:100] + "...")
    except Exception as e:
        print(f"Failed to load best model: {str(e)}")

# Create model loading example script
with open("load_models_example.py", "w", encoding="utf-8") as f:
    f.write('''# Model loading example
# ============================================

from pycaret.classification import load_model,predict_model
import pandas as pd

# Load the best model
best_model=load_model("best_model")

# Predict using new data (DataFrame format)
# predictions=predict_model(best_model,data=new_data)

# Load specific models
# model_01=load_model("model_01_LogisticRegression")
# model_02=load_model("model_02_RidgeClassifier")

# Batch load all models
import glob
models={}
for model_file in glob.glob("model_*.pkl"):
    model_name=model_file.replace(".pkl","")
    models[model_name]=load_model(model_file)
    print(f"Loaded model: {model_name}")

print("\\nAll models loaded successfully!")
''')

print("Model loading example saved to load_models_example.py")

print("\n" + "="*50)
print("All model saving operations completed!")
print("="*50)
print("""
Summary:
1. All 16 models saved as independent .pkl files
2. Best model saved separately as best_model.pkl
3. Model list and details saved to text/CSV files
4. All files packaged into all_models.zip
5. Example code provided for loading models

You can:
1. Use load_model('model_filename') to load any model
2. Check models_summary.csv for model details
3. Refer to load_models_example.py for usage examples
""")


开始保存模型...
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  1: RidgeClassifier -> model_01_RidgeClassifier.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  2: LinearDiscriminantAnalysis -> model_02_LinearDiscriminantAnalysis.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  3: CatBoostClassifier -> model_03_CatBoostClassifier.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  4: LogisticRegression -> model_04_LogisticRegression.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  5: ExtraTreesClassifier -> model_05_ExtraTreesClassifier.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  6: XGBClassifier -> model_06_XGBClassifier.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  7: SGDClassifier -> model_07_SGDClassifier.pkl
Transformation Pipeline and Model Successfully Saved
✓ 已保存模型  8: LGBMClassifier -> model_08_LGBMClassifier.pkl
Transformation Pipeline and Model Successfully Save

In [ ]:
import pandas as pd

all_rows=[]

# Evaluate each model on the test set and export metrics to CSV
for i,model in enumerate(top16_model,start=1):
    model_name=model.__class__.__name__
    print(f"\nModel {i}: {model_name}")
    try:
        # Predict/evaluate on the test set
        _=predict_model(model,data=exp.test)

        # Retrieve evaluation metrics for this prediction
        test_results=pull()  # Must follow predict_model inside the loop

        # Ensure the result is a DataFrame
        if not isinstance(test_results,pd.DataFrame):
            test_results=pd.DataFrame(test_results)

        # Add model information for aggregation
        row=test_results.copy()
        row.insert(0,"ModelIndex",i)
        row.insert(1,"ModelName",model_name)

        all_rows.append(row)

        print(test_results)

    except Exception as e:
        # Record failure to avoid interruption
        all_rows.append(pd.DataFrame([{
            "ModelIndex":i,
            "ModelName":model_name,
            "Error":str(e)
        }]))
        print(f"Model {i} ({model_name}) evaluation failed: {e}")

# Aggregate and export
final_df=pd.concat(all_rows,ignore_index=True)
final_df.to_csv("mincancer1_top16_test_metrics.csv",index=False,encoding="utf-8-sig")

print("\nExported: mincancer1_top16_test_metrics.csv")
print(final_df.head())


模型 1: RidgeClassifier


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,bal_acc,f1_macro
0,Ridge Classifier,0.9730,0,0.9730,0.9730,0.9729,0.9654,0.9654,0.9568,0.9561


              Model  Accuracy  AUC  Recall  Prec.      F1   Kappa     MCC  \
0  Ridge Classifier     0.973    0   0.973  0.973  0.9729  0.9654  0.9654   

   bal_acc  f1_macro  
0   0.9568    0.9561  

模型 2: LinearDiscriminantAnalysis


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,bal_acc,f1_macro
0,Linear Discriminant Analysis,0.9747,0.9990,0.9747,0.9754,0.9748,0.9676,0.9676,0.9705,0.9588


                          Model  Accuracy    AUC  Recall   Prec.      F1  \
0  Linear Discriminant Analysis    0.9747  0.999  0.9747  0.9754  0.9748   

    Kappa     MCC  bal_acc  f1_macro  
0  0.9676  0.9676   0.9705    0.9588  

模型 3: CatBoostClassifier


...

                             Model  Accuracy     AUC  Recall   Prec.      F1  \
0  Quadratic Discriminant Analysis    0.8767  0.9494  0.8767  0.8167  0.8413   

    Kappa     MCC  bal_acc  f1_macro  
0  0.8398  0.8452   0.6623    0.6333  

模型 15: AdaBoostClassifier


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,bal_acc,f1_macro
0,Ada Boost Classifier,0.7584,0.9198,0.7584,0.7393,0.7312,0.6903,0.7034,0.6366,0.6076


                  Model  Accuracy     AUC  Recall   Prec.      F1   Kappa  \
0  Ada Boost Classifier    0.7584  0.9198  0.7584  0.7393  0.7312  0.6903   

      MCC  bal_acc  f1_macro  
0  0.7034   0.6366    0.6076  

模型 16: DummyClassifier


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,bal_acc,f1_macro
0,Dummy Classifier,0.3699,0,0.3699,0.1369,0.1998,0.0000,0.0000,0.1429,0.0772


              Model  Accuracy  AUC  Recall   Prec.      F1  Kappa  MCC  \
0  Dummy Classifier    0.3699    0  0.3699  0.1369  0.1998    0.0  0.0   

   bal_acc  f1_macro  
0   0.1429    0.0772  

已导出：mincancer1_top16_test_metrics.csv
   ModelIndex                   ModelName                         Model  \
0           1             RidgeClassifier              Ridge Classifier   
1           2  LinearDiscriminantAnalysis  Linear Discriminant Analysis   
2           3          CatBoostClassifier           CatBoost Classifier   
3           4          LogisticRegression           Logistic Regression   
4           5        ExtraTreesClassifier        Extra Trees Classifier   

   Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC  bal_acc  f1_macro  
0    0.9730  0.0000  0.9730  0.9730  0.9729  0.9654  0.9654   0.9568    0.9561  
1    0.9747  0.9990  0.9747  0.9754  0.9748  0.9676  0.9676   0.9705    0.9588  
2    0.9730  0.9991  0.9730  0.9732  0.9729  0.9654  0.9655   0.9682    

In [ ]:
save_pycaret_data_for_tabpfn(exp, "mincancer1_pycaret_tabpfn_splits")

| Category             | Parameter              | Value           |
| -------------------- | ---------------------- | --------------- |
| Framework            | PyCaret version        | 3.0.0           |
| Randomization        | Session ID             | 2026            |
| Train/Test split     | Train size             | 0.8             |
| Cross-validation     | Fold number            | 5               |
| CV method            | Fold generator         | StratifiedKFold |
| Preprocessing        | Numeric imputation     | mean            |
| Preprocessing        | Categorical imputation | mode            |
| Preprocessing        | Normalization          | True            |
| Normalization method | zscore                 |                 |
| GPU usage            | False                  |                 |


| Category             | Parameter              | Value           |
| -------------------- | ---------------------- | --------------- |
| Framework            | PyCaret version        | 3.0.0           |
| Randomization        | Session ID             | 2026            |
| Dataset              | Total samples          | 2959            |
| Dataset              | Training samples       | 2367            |
| Dataset              | Test samples           | 592             |
| Dataset              | Features               | 144             |
| Dataset              | Classes                | 7               |
| Train/Test split     | Train size             | 0.8             |
| Cross-validation     | Fold number            | 5               |
| Cross-validation     | Method                 | StratifiedKFold |
| Preprocessing        | Numeric imputation     | mean            |
| Preprocessing        | Categorical imputation | mode            |
| Preprocessing        | Feature normalization  | True            |
| Normalization method | zscore                 |                 |
| Hardware             | GPU usage              | False           |
